In [0]:
%sql create schema if not exists apexlife.silver;

In [0]:
%sql select * from apexlife.bronze.patients_raw ;

patient_id,first_name,last_name,gender,dob,city,_rescued_data
P001,Aarav,Sharma,M,1980-02-14,Mumbai,null
P002,Riya,Verma,F,1975-07-22,Delhi,null
P003,Kabir,Menon,M,1990-11-02,Bangalore,null
P004,Sneha,Rao,F,1988-09-19,Hyderabad,null
P005,Vikram,Singh,M,1965-05-05,Chennai,null


In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = 'apexlife.bronze.patients_raw'
silver_table = 'apexlife.silver.dim_patient'

checkpoint_path  = "abfss://data@apexlife.dfs.core.windows.net/silver/dim_patient/checkpoint/"

In [0]:
df = (
    spark.readStream.table(bronze_table)
)

In [0]:
df = (
    df
    .dropDuplicates(['patient_id'])
    .withColumn("patient_first_last_name_masked", # Hashing first and last name
                sha2(
                    concat_ws('|',col("first_name"),col("last_name")), 256)
                )
    .withColumn('load_time' , current_timestamp())
)

In [0]:
from delta.tables import DeltaTable

In [0]:
def merge_dim_patient(batch_df , batch_id) :
    if not spark.catalog.tableExists(silver_table) :
        batch_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)
        return 
    
    dim_patient = DeltaTable.forName(spark , silver_table)

    (
        dim_patient.alias('t').merge(
        batch_df.alias('s'),
        't.patient_id = s.patient_id'
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )

(
    df.writeStream
    .foreachBatch(merge_dim_patient)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

In [0]:
%sql select * from apexlife.silver.dim_patient ;

patient_id,first_name,last_name,gender,dob,city,_rescued_data,patient_first_last_name_masked,load_time
P001,Aarav,Sharma,M,1980-02-14,Mumbai,null,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,2026-06-04T14:26:06.004Z
P003,Kabir,Menon,M,1990-11-02,Bangalore,null,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,2026-06-04T14:26:06.004Z
P002,Riya,Verma,F,1975-07-22,Delhi,null,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,2026-06-04T14:26:06.004Z
P005,Vikram,Singh,M,1965-05-05,Chennai,null,deb97260b70ff54088f4b6cf6f75edf041b65eed5b26c608290dabc0e9efe0b6,2026-06-04T14:26:06.004Z
P004,Sneha,Rao,F,1988-09-19,Hyderabad,null,78a3e0a772632903fef2ffe51d6119d7df94f79dfb3295be92ecafe5029dc20f,2026-06-04T14:26:06.004Z
